# Deliverable 2

In [21]:
# Imports

import re
from pathlib import Path
from charset_normalizer import from_path

#### Task 1

In [22]:
test_path = Path("data/2026/student_union/POS_01/register_sales.log")

def file_reader(file_path: Path) -> tuple[str, str]:
    """
    Reads a file checking its encoding.

    args:
        file_path: The path for the file to be checked
    
    returns:
        text: the encoded text
        encoding: the encoding of the given text

    """
    raw_bytes = Path(file_path).read_bytes()

    # Checking for utf-8 encoding
    if raw_bytes[:3] == b"\xef\xbb\xbf":
        encoding = "utf-8-sig"

    # Otherwise try to guess correct encoding
    else:
        best_match = from_path(file_path).best()
        if best_match is None:
            raise ValueError(f"Could not detect encoding of {file_path}")
        encoding = best_match.encoding

    decoded_text = raw_bytes.decode(encoding)
    return decoded_text, encoding

text, encoding = file_reader(test_path)
print(f"Encoding: {encoding}\n{text}")

Encoding: ascii
[2026-10-15 07:02:16] [POS_01] item=Croissant price=4.00 EUR (SUCCESS)
[2026-10-15 07:04:24] [POS_01] item=Muffin price=3.38 EUR (CANCELLED)
[2026-10-15 07:04:35] [POS_01] item=Espresso price=2.94 USD (SUCCESS)
[2026-10-15 07:05:04] [POS_01] item=Espresso price=2.96 USD (CANCELLED)
[2026-10-15 07:05:14] [POS_01] item=Espresso price=2.79 EUR (SUCCESS)
[2026-10-15 07:06:43] [POS_01] item=Latte price=4.20 EUR (SUCCESS)
[DEBUG] BARCODE SCANNER REBOOTED
[2026-10-15 07:09:14] [POS_01] item=Cappuccino price=4.23 EUR (REFUND)
[2026-10-15 07:09:59] [POS_01] item=Sandwich price=7.08 NOK (SUCCESS)
[2026-10-15 07:10:24] [POS_01] item=Latte price=4.26 EUR (SUCCESS)
=== NETWORK SYNC OK ===
[2026-10-15 07:11:58] [POS_01] item=Muffin price=3.02 EUR (SUCCESS)
--- PRINTER PAPER LOW ---
[2026-10-15 07:12:34] [POS_01] item=Sandwich price=7.78 USD (SUCCESS)
[2026-10-15 07:14:00] [POS_01] item=Muffin price=3.10 NOK (SUCCESS)
[2026-10-15 07:16:28] [POS_01] item=Muffin price=3.98 NOK (SUCCESS)

For de log_parser() I've used lecture 6 for regex syntax

In [ ]:
def log_parser(decoded_text: str) -> list[dict][str, str | float]:
    """
    Uses regular expression to capture 6 key fields from valid lines in a log file and
    ignores noise lines.

    Key fields:
        timestamp
        register_id
        item
        price
        currency
        status
    
    args:
        decoded_text: a string of from decoded file.

    returns: 
        list: List of dictonaries with valid transaction record
    """

    verbose_pattern = re.compile(
        r"""                                   
        \[(?P<timestamp> \d{4}-\d{2}-\d{2}\ \d{2}:\d{2}:\d{2})\]    # datetime YYYY-MM-DD HH:MM:SS
        \s+                                                         # whitespace
        \[(?P<register_id> POS_\d{2})\]                             # POS_xx
        \s+                                                         # whitespace
        item=(?P<item>\S+)                                          # item
        \s+                                                         # whitespace
        price=(?P<price>\S+)                                        # price
        \s+                                                         # whitespace
        (?P<currency>[A-Za-z]{3})                                   # currency
        \s+                                                         # whitespace
        \( (?P<status>[\w-]+) \)                                    # status
        $                                   
    """,
        re.VERBOSE,
    )    

    valid_records = []
    for line in decoded_text.splitlines():
        m = verbose_pattern.match(line)

        # check for non-valid lines
        if not m:
            continue

        valid_records.append(
            {
            "timestamp": m.group("timestamp"),
            "register_id": m.group("register_id"),
            "item": m.group("item"),
            "price": float(m.group("price")), # converts str to float here
            "currency": m.group("currency").upper(),
            "status": m.group("status"),
            }
        )

    return valid_records

log_parser(text)


    

[{'timestamp': '2026-10-15 07:02:16',
  'register_id': 'POS_01',
  'item': 'Croissant',
  'price': 4.0,
  'currency': 'EUR',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:04:24',
  'register_id': 'POS_01',
  'item': 'Muffin',
  'price': 3.38,
  'currency': 'EUR',
  'status': 'CANCELLED'},
 {'timestamp': '2026-10-15 07:04:35',
  'register_id': 'POS_01',
  'item': 'Espresso',
  'price': 2.94,
  'currency': 'USD',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:05:04',
  'register_id': 'POS_01',
  'item': 'Espresso',
  'price': 2.96,
  'currency': 'USD',
  'status': 'CANCELLED'},
 {'timestamp': '2026-10-15 07:05:14',
  'register_id': 'POS_01',
  'item': 'Espresso',
  'price': 2.79,
  'currency': 'EUR',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:06:43',
  'register_id': 'POS_01',
  'item': 'Latte',
  'price': 4.2,
  'currency': 'EUR',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:09:14',
  'register_id': 'POS_01',
  'item': 'Cappuccino',
  'price': 4.23,
  

#### Task 2

#### Task 3

#### Task 4

#### Task 5